# Clean TAPR Data - Apostrophes
created 2/20/25  
updated 6/17/25  

This notebook builds on and replaces "Editing campus..." and "delete_apostrophes". 
It should do the following:  
* remove leading apostrophes
* fill in leading zeroes for entity identifiers (campus, district, region)
* capitalize column headers 
 
6/17/25 cleaned separate accountability ratings for SY2022-23

### using this notebook 
Before getting started, cd into scuole-data, then...  
```pipenv shell```

## Settings

In [6]:
school_year = '2022-2023'

In [7]:
#%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 11.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 10.4 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup

In [1]:
# import libraries you might need
import pandas as pd
import os.path

In [ ]:
levels = [
  "campus",
  "district",
  "region",
  "state"
]

# SY2023-24 no A-F assessment, per pending lawsuit
files = [
    "accountability.csv",
    "ap-ib-sat-act.csv",
    "attendance.csv",
    "longitudinal-rate.csv",
    "postsecondary-readiness-and-non-staar-performance-indicators.csv",
    "reference.csv",
    "staff-and-student-information.csv"
]

column_names = [
  "DISTRICT",
  "COUNTY",
  "REGION",
  "CAMPUS"
]

## Data Cleaning

In [7]:
print(f"Cleaning TAPR file...{school_year}\nLEVEL | FILE | [COLUMNS] ... to edit")

for level in levels:
  for file in files:

    # set the data path    
    path = "../tapr/" + school_year + "/" + level + "/" + file
    
    # check if the file exists and proceed only if it does
    if os.path.exists(path):
        data = pd.read_csv(path)
        
        data.columns = data.columns.str.upper()

        # see if the data has 'DISTRICT', 'COUNTY' or 'REGION' column
        s = pd.Series(column_names).isin(data.columns)
        indexes = list(s[s].index)
        columns_to_edit = [column_names[i] for i in indexes]

        print(f"{level} | {file} | {columns_to_edit}")

        # if the data has the column(s), get rid of the apostrophes
        if len(columns_to_edit) > 0:
            for column in columns_to_edit:
              data[column] = data[column].apply(lambda x: str(x).replace("'", ''))
              data[column] = data[column].astype(str)
              
              # add leading zeroes
              if column == 'REGION':
                data[column] = data[column].apply(lambda x: str(x).zfill(2))
              elif column == 'COUNTY':
                data[column] = data[column].apply(lambda x: str(x).zfill(3)) 
              elif column == 'DISTRICT':
                data[column] = data[column].apply(lambda x: str(x).zfill(6))
              elif column == 'CAMPUS':
                data[column] = data[column].apply(lambda x: str(x).zfill(9))

            # update the data
            data.to_csv(path, index = False)

print("Cleaning routine completed!")

Cleaning TAPR file...2022-2023
LEVEL | FILE | [COLUMNS] ... to edit
campus | accountability.csv | ['DISTRICT', 'CAMPUS']
district | accountability.csv | ['DISTRICT']
Cleaning routine completed!
